# LangGraph — stateful graphs, supervisor pattern

LangGraph models the agent as a *graph*:

* **Nodes** are typed functions that receive the shared state and return a partial-state delta.
* **Edges** connect nodes. **Conditional edges** route based on state — that's where the agency lives.
* A **checkpointer** snapshots state after every node, enabling pause / resume / replay (the foundation for HITL).

This notebook shows the supervisor pattern: one routing node decides whether to call `search`, `cite`, or finish.

## The canonical code (paste into a project with `langgraph` installed)

```python
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from task import search_corpus, fetch_paper, cite

class AgentState(TypedDict):
    question: str
    hits: list
    papers: list
    answer: str | None
    history: Annotated[list, add]

def supervisor(state: AgentState) -> dict:
    if not state.get('hits'):
        return {'history': ['route -> search']}
    if not state.get('answer'):
        return {'history': ['route -> synthesise']}
    return {'history': ['route -> cite']}

def search_node(state: AgentState) -> dict:
    hits = search_corpus(state['question'], k=2)
    return {'hits': hits, 'papers': [fetch_paper(h['arxiv_id']) for h in hits]}

def synthesise_node(state: AgentState) -> dict:
    return {'answer': f"[{state['hits'][0]['arxiv_id']}] {state['hits'][0]['snippet']}"}

def cite_node(state: AgentState) -> dict:
    return {'history': [cite(state['hits'][0]['arxiv_id'], state['answer'])]}

def route(state: AgentState) -> str:
    if not state.get('hits'): return 'search'
    if not state.get('answer'): return 'synthesise'
    return 'cite'

graph = StateGraph(AgentState)
graph.add_node('supervisor', supervisor)
graph.add_node('search', search_node)
graph.add_node('synthesise', synthesise_node)
graph.add_node('cite', cite_node)
graph.add_edge(START, 'supervisor')
graph.add_conditional_edges('supervisor', route, {'search':'search','synthesise':'synthesise','cite':'cite'})
graph.add_edge('search', 'supervisor')
graph.add_edge('synthesise', 'supervisor')
graph.add_edge('cite', END)
app = graph.compile(checkpointer=MemorySaver())
result = app.invoke({'question': 'What is RA-MoE?'}, config={'configurable':{'thread_id':'demo'}})
```

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import get_question, run_evaluation
from eval import langgraph_solve
trace = langgraph_solve(get_question('q01'))
for step in trace.steps:
    print(f"{step.role}: {step.name or ''}  {step.output_summary or step.content or ''}"[:140])

## Why the graph wrapping is worth it

* **Typed state** — every node receives the same `AgentState` TypedDict; misuse is a type error, not a runtime bug.
* **Conditional edges** — the routing decision is *data* (a string returned by `route`), not control flow buried in an `if` chain.
* **Checkpointer** — `app.invoke({...}, config={'thread_id': 'x'})` persists state per thread, so the next call resumes exactly where the previous one stopped. This is what makes `interrupt()` / human approval possible (see `04-human-in-the-loop/`).
* **`stream(...)` for free** — one method call gives you token-by-token streaming and per-node deltas, both with the same checkpointer guarantees.